# Assignment 9: Hugging Face Transformers for Slovak NLP — Classification, Model Benchmarking, Question Answering & Error Analysis
**Author:** Kristian Hudák  
**Course:** AI Olympiad (OAI) Preparation Course

This project investigates state-of-the-art Transformer architectures for the Slovak language:
1. **Task 1: Slovak Text Classification** — Fine-tuning `gerulata/slovakbert` for hate speech detection on the `TUKE-KEMT/hate_speech_slovak` dataset.
2. **Task 2: Multi-Model Benchmark** — Comparative evaluation of language-specific vs multilingual models:
   - `gerulata/slovakbert` (~125M parameters)
   - `xlm-roberta-base` (~278M parameters, 100 languages)
   - `distilbert-base-multilingual-cased` (~134M parameters)
3. **Task 3: Slovak Question Answering** — Fine-tuning and evaluating extractive QA models on the `SK-QuAD` dataset (91k+ QA pairs).
4. **Task 4: Qualitative & Quantitative Error Analysis** — Systematic taxonomy of model failures and concrete mitigation strategies.


> **Computational Note:** Model fine-tuning was performed in Google Colab GPU sessions with checkpoint caching due to platform compute limits.


In [ ]:
import json
import numpy as np
import torch
from matplotlib import pyplot as plt


from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    pipeline
)
from datasets import load_dataset, DatasetDict

---

## Task 1: Slovak Text Classification (1 point)

Train a hate speech detector for Slovak social media using the [`hate_speech_slovak`](https://huggingface.co/datasets/TUKE-KEMT/hate_speech_slovak) dataset created by the Technical University of Košice.

### Instructions:

1. Load the dataset and explore its structure (how many examples? what are the labels?)
2. Fine-tune [`gerulata/slovakbert`](https://huggingface.co/gerulata/slovakbert) on this binary classification task
3. Report accuracy and F1 score on the test set
4. Try a few example sentences of your own - does the model work as expected?

In [ ]:
# Task 1.1: Load and explore the dataset
# Your code here
import pandas as pd

splits = {'train': 'train.json', 'test': 'test.json'}
df = pd.read_json("hf://datasets/TUKE-KEMT/hate_speech_slovak/" + splits["train"], lines=True)


In [ ]:
# Task 1.1.1
print(df.head(8))
print('-'*5)
print(df['label'])
print('-'*5)
print(df.label.value_counts())
print('-'*5)
print(f"Dataset shape (rows, columns): {df.shape}")
print(f"Number of examples: {len(df)}")

   id                                               text  label
0   1  poliaci maju viac rozumu ako slováci namyslení...      1
1   2  user19773 tak ja uz na dovolenke a kaudy den o...      0
2   3        user8371 ak by vám to neprechádzalo tu mate      0
3   4  user13335 nezaslúžia, ale o tom nie je reč. di...      0
4   5  user35231 obyčajní chrapúni bez kúska sebarefl...      1
5   6                ked sa pyta ciganikova na vzdelanie      0
6   7                                   user36201 uvidis      0
7   8  user24836 v tomto prípade ano,inak by nemali h...      0
-----
0        1
1        0
2        0
3        0
4        1
        ..
11865    0
11866    0
11867    1
11868    0
11869    1
Name: label, Length: 11870, dtype: int64
-----
label
0    8625
1    3245
Name: count, dtype: int64
-----
Dataset shape (rows, columns): (11870, 3)
Number of examples: 11870


In [ ]:
from datasets import load_dataset

ds = load_dataset("TUKE-KEMT/hate_speech_slovak")

In [ ]:
# Task 1.2: Fine-tune SlovakBERT
# Your code here
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification # Changed to AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("gerulata/slovakbert")
model = AutoModelForSequenceClassification.from_pretrained("gerulata/slovakbert", num_labels=2) # num_labels=2 for binary classification

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at gerulata/slovakbert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from datasets import load_dataset, DatasetDict, Dataset



Tuke_dataset = DatasetDict(
    train=ds['train'].shuffle(seed=1111),
    val=ds['test'].shuffle(seed=1111),
)

In [ ]:
Tuke_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 11870
    })
    val: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1319
    })
})

In [ ]:
Tuke_dataset['train'][:10]

{'id': [10578, 902, 2217, 6272, 4878, 9781, 1853, 3258, 12668, 13009],
 'text': ['chudáci ruzzofilovia ,, to za brat za braťa ,, im nejak nevychádza,,, niekto tu ( o.eba..) mimochodom našimi bratmi nie sú češi ..',
  'user31279 obrovský úspech.. .. keď už si aj tabák chváli tento "prestup" tak to bude určite budúce fiasko..',
  'veľa zdravia, gratulujem',
  'user38414 prečo by si sa mal stať jednobunkovcom keď ním si! aha to si nevedel?! tak už vieš!',
  'seci normálne ludjjom potrebuje miér slobodu',
  'user21793 to ja viem. ľudia sa boja nepoznaného alebo exotického a médiá ich v tom veselo podporujú.',
  'hned ako to vyhrate kokota machaja vyjebte na ulicu',
  'ty čuračik,,pravda je taka že smer, je banda skorumpovana',
  'user38413 aďka primitivizmus kraľuje slovenskému národu že  vidieť to krásne v komentároch ešte že majko bude mať menej percent ako odtučnené mlieko',
  'už tam mal byt dávno.'],
 'label': [0, 0, 0, 1, 0, 0, 1, 1, 0, 0]}

In [ ]:
tokenized_Tuke_dataset = Tuke_dataset.map(
    lambda example: tokenizer(example['text'], padding=True, truncation=True),
    batched=True,
    batch_size=16
)
tokenized_Tuke_dataset = tokenized_Tuke_dataset.remove_columns(["text", "id"])
tokenized_Tuke_dataset = tokenized_Tuke_dataset.rename_column("label", "labels")
tokenized_Tuke_dataset.set_format("torch")

Map:   0%|          | 0/11870 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
tokenized_Tuke_dataset['train'][0:2]

{'labels': tensor([0, 0]),
 'input_ids': tensor([[    0, 40699,  6097,   760,   100,  1521,  3883,  3224,    84,     4,
             11,    23,  2701,    23,  3580,  3600,    84,     4,   187,  2789,
            487,  4753,     4,     4,     4,   327,    58,  1182, 50189,    17,
              5,   179,   336,     5,     5, 50191, 12011,  8577,  2701,    63,
             37,    31,  2106,   742,    44,     5,     2,     1,     1,     1,
              1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
              1,     1,     1,     1],
         [    0, 20352,   259,  1260, 13086,  6837,  3252,     5,     5,    44,
              5,    57,    34,    14,    16, 31288,  1281, 45481,   210,  1182,
          50250,  1210,  2136, 50250,    25,    11,    48,   629, 15082, 11863,
            499,     5,     5,     2,     1,     1,     1,     1,     1,     1,
              1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
              1,     1,     1,     1,    

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

arguments = TrainingArguments(
    output_dir="sample_hf_trainer",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epoch s=5,
    eval_strategy="epoch", # run validation at the end of each epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    seed=224,
    report_to="tensorboard"
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # calculates the accuracy
    return {"accuracy": accuracy_score(labels, predictions),
            "f1": f1_score(labels, predictions)
            }


trainer = Trainer(
    model=model,
    args=arguments,
    train_dataset=tokenized_Tuke_dataset['train'],
    eval_dataset=tokenized_Tuke_dataset['val'], # change to test when you do your final evaluation!
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)



/tmp/ipython-input-2657186839.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.491000,0.466960,0.809704,0.563478
2,0.434900,0.471463,0.790751,0.619835
3,0.331900,0.557273,0.778620,0.618799
4,0.304800,0.617854,0.787718,0.602273
5,0.210800,0.701683,0.787718,0.616438


TrainOutput(global_step=3710, training_loss=0.34250665649249545, metrics={'train_runtime': 3000.3435, 'train_samples_per_second': 19.781, 'train_steps_per_second': 1.237, 'total_flos': 8007213525558240.0, 'train_loss': 0.34250665649249545, 'epoch': 5.0})

In [ ]:
results = trainer.predict(tokenized_Tuke_dataset['val'])
results

PredictionOutput(predictions=array([[ 0.7618124, -1.1696694],
       [ 1.1650157, -1.4407855],
       [ 1.1617795, -1.3918322],
       ...,
       [ 1.0690004, -1.4140216],
       [ 1.1485784, -1.4214267],
       [ 1.1552596, -1.4003452]], dtype=float32), label_ids=array([0, 0, 0, ..., 0, 0, 0]), metrics={'test_loss': 0.46695956587791443, 'test_accuracy': 0.8097043214556482, 'test_f1': 0.5634782608695652, 'test_runtime': 7.0723, 'test_samples_per_second': 186.501, 'test_steps_per_second': 11.736})

In [ ]:
# Task 1.3: Report accuracy and F1 score on test set
accuracy_value = results.metrics['test_accuracy']
f1_value = results.metrics['test_f1']

print(f"Najlepšie výsledky slovakbertu na hate speech na validačnej sade(podla accuracy): Accuracy = {accuracy_value:.4f}, F1-Score = {f1_value:.4f}")

In [ ]:
# Task 1.4: Try your own example sentences
# Your code here
import torch
import os
test_str = "Maros je debil ale dobry hudobnik !"

# Load from the specific checkpoint to ensure the correct configuration is picked up
finetuned_model = AutoModelForSequenceClassification.from_pretrained("sample_hf_trainer/checkpoint-2968")
model_inputs = tokenizer(test_str, return_tensors="pt")
prediction = torch.argmax(finetuned_model(**model_inputs).logits)
print('NEGATIVE= nenachadza sa tam hate speech, POSITIVE= nachadza sa tam hate speech')
print(["NEGATIVE", "POSITIVE"][prediction])

---

## Task 2: Model Comparison (2 points)

Compare three different approaches on the same Slovak hate speech dataset:

1. `gerulata/slovakbert` (Slovak-specific, ~125M parameters)
2. `xlm-roberta-base` (multilingual, 100 languages, ~278M parameters)
3. `distilbert-base-multilingual-cased` (multilingual, smaller, ~134M parameters)

### Questions to Consider:

Which model performs best? Why might a Slovak-specific model outperform larger multilingual ones? Consider:
- Training data composition
- Vocabulary coverage for Slovak characters (ä, č, ď, é, í, ĺ, ľ, ň, ó, ô, ŕ, š, ť, ú, ý, ž)
- Model size vs. language specificity tradeoff

In [ ]:
# Task 2.1: Train and evaluate SlovakBERT (if not already done in Task 1)
# Your code here
results1 = trainer.predict(tokenized_Tuke_dataset['val'])
results1

PredictionOutput(predictions=array([[ 0.7618124, -1.1696694],
       [ 1.1650157, -1.4407855],
       [ 1.1617795, -1.3918322],
       ...,
       [ 1.0690004, -1.4140216],
       [ 1.1485784, -1.4214267],
       [ 1.1552596, -1.4003452]], dtype=float32), label_ids=array([0, 0, 0, ..., 0, 0, 0]), metrics={'test_loss': 0.46695956587791443, 'test_accuracy': 0.8097043214556482, 'test_f1': 0.5634782608695652, 'test_runtime': 8.0856, 'test_samples_per_second': 163.13, 'test_steps_per_second': 10.265})

In [ ]:
print(f"Best SlovakBERT validation results on hate speech (by accuracy): Accuracy = {accuracy_value:.4f}, F1-Score = {f1_value:.4f}")


In [ ]:
# Task 2.2: Train and evaluate XLM-RoBERTa
# Your code here
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

tokenizer2 = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
model2 = AutoModelForSequenceClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=2)

arguments = TrainingArguments(
    output_dir="sample_hf_trainer",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch", # run validation at the end of each epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    seed=224,
    report_to="tensorboard"
)


def compute_metrics(eval_pred):
    """Called at the end of validation. Gives accuracy"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # calculates the accuracy
    return {"accuracy": accuracy_score(labels, predictions),
            "f1": f1_score(labels, predictions)
            }


trainer2 = Trainer(
    model=model2,
    args=arguments,
    train_dataset=tokenized_Tuke_dataset['train'],
    eval_dataset=tokenized_Tuke_dataset['val'], # change to test when you do your final evaluation!
    tokenizer=tokenizer2,
    compute_metrics=compute_metrics
)

trainer2.train()

results2 = trainer2.predict(tokenized_Tuke_dataset['val'])
results2



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-4244770208.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer2 = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.597800,0.576550,0.727066,0.000000
2,0.587300,0.570483,0.727066,0.000000
3,0.573500,0.579076,0.727066,0.000000
4,0.585300,0.573801,0.727066,0.000000
5,0.572300,0.561063,0.729340,0.118519


PredictionOutput(predictions=array([[ 0.14511792, -0.37832275],
       [ 0.6057883 , -0.81172395],
       [ 0.6171259 , -0.8097384 ],
       ...,
       [ 0.4327635 , -0.7232987 ],
       [ 0.6241009 , -0.8070314 ],
       [ 0.62253416, -0.80794317]], dtype=float32), label_ids=array([0, 0, 0, ..., 0, 0, 0]), metrics={'test_loss': 0.5610631704330444, 'test_accuracy': 0.7293404094010614, 'test_f1': 0.11851851851851852, 'test_runtime': 7.017, 'test_samples_per_second': 187.972, 'test_steps_per_second': 11.828})

In [ ]:
accuracy_value1 = results2.metrics['test_accuracy']
f1_value1 = results2.metrics['test_f1']

print(f"Najlepšie výsledky slovakbertu na hate speech na validačnej sade(podla accuracy): Accuracy = {accuracy_value1:.4f}, F1-Score = {f1_value1:.4f}")

In [ ]:
# Task 2.3: Train and evaluate multilingual DistilBERT
# Your code here
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np

tokenizer3 = AutoTokenizer.from_pretrained("distilbert/distilbert-base-multilingual-cased")
model3 = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-multilingual-cased",num_labels=2)

from sklearn.metrics import accuracy_score, f1_score

arguments = TrainingArguments(
    output_dir="sample_hf_trainer",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch", # run validation at the end of each epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    seed=224,
    report_to="tensorboard"
)


def compute_metrics(eval_pred):
    """Called at the end of validation. Gives accuracy"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # calculates the accuracy
    return {"accuracy": accuracy_score(labels, predictions),
            "f1": f1_score(labels, predictions)
            }


trainer3 = Trainer(
    model=model3,
    args=arguments,
    train_dataset=tokenized_Tuke_dataset['train'],
    eval_dataset=tokenized_Tuke_dataset['val'], # change to test when you do your final evaluation!
    tokenizer=tokenizer3,
    compute_metrics=compute_metrics
)

trainer3.train()

results3 = trainer3.predict(tokenized_Tuke_dataset['val'])
results3

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2860953864.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer3 = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.585100,0.569615,0.727066,0.000000
2,0.560000,0.560372,0.709629,0.400626
3,0.464000,0.608730,0.675512,0.488038
4,0.416100,0.661402,0.703563,0.442225
5,0.291000,0.741446,0.712661,0.428356


PredictionOutput(predictions=array([[ 0.43775502, -0.49360067],
       [ 0.5958747 , -0.7431423 ],
       [ 1.2015704 , -1.40808   ],
       ...,
       [ 0.21245432, -0.2548218 ],
       [ 0.16325407, -0.18799222],
       [ 0.7245565 , -0.8642736 ]], dtype=float32), label_ids=array([0, 0, 0, ..., 0, 0, 0]), metrics={'test_loss': 0.5603721737861633, 'test_accuracy': 0.709628506444276, 'test_f1': 0.40062597809076683, 'test_runtime': 3.8219, 'test_samples_per_second': 345.119, 'test_steps_per_second': 21.717})

In [ ]:
accuracy_value3 = results3.metrics['test_accuracy']
f1_value3 = results3.metrics['test_f1']

print(f"Najlepšie výsledky slovakbertu na hate speech na validačnej sade(podla accuracy): Accuracy = {accuracy_value:.4f}, F1-Score = {f1_value:.4f}")

In [ ]:
# Task 2.4: Compare results and analyze
# Your code here
print(f" Najlepšie výsledky slovakbertu na hate speech na validačnej sade(podla accuracy): Accuracy = {accuracy_value:.4f}, F1-Score = {f1_value:.4f}")
print(f" Najlepšie výsledky xml-roberta na hate speech na validačnej sade(podla accuracy): Accuracy = {accuracy_value2:.4f}, F1-Score = {f1_value2:.4f}")
print(f" Najlepšie výsledky destilbert na hate speech na validačnej sade(podla accuracy): Accuracy = {accuracy_value3:.4f}, F1-Score = {f1_value3:.4f}")

### Model Performance Analysis for Slovak Hate Speech Detection

Following fine-tuning across all candidate models, the domain-specialized **Fine-tuned SlovakBERT (`gerulata/slovakbert`)** achieved the highest overall performance, outperforming both the larger multilingual `xlm-roberta-base` (~278M params) and the compact `distilbert-base-multilingual` (~134M params).

**Key Drivers of Specialization Advantage:**
1. **Deep Monolingual Language Modeling:** SlovakBERT was pre-trained exclusively on extensive Slovak web text and corpora. In contrast, XLM-RoBERTa partitions its vocabulary and parameter capacity across 100 distinct languages. SlovakBERT possesses a much richer internal representation of Slovak morphological inflections (declensions, conjugations, grammatical cases) and colloquial vocabulary.
2. **Superior Context Discrimination:** Due to dedicated Slovak tokenization (byte-level BPE calibrated for Slovak accented characters `á, č, ď, é, í, ľ, ň, ó, ô, ŕ, š, ť, ú, ý, ž`), SlovakBERT splits words into fewer fragmented subwords. This preserves morphological integrity, crucial for distinguishing subtle irony or implicit offensive language.
3. **Resource Efficiency & Signal-to-Noise Ratio:** By training exclusively on Slovak, cross-lingual negative interference and vocabulary dilution are eliminated, yielding faster convergence and higher validation accuracy and F1 scores.


---

## Task 3: Slovak Question Answering (2 points)

Use the [`SK-QuAD`](https://huggingface.co/datasets/TUKE-DeutscheTelekom/skquad) dataset (91,000+ QA pairs) to:

1. Explore the dataset - what types of questions are included?
2. Fine-tune a QA model on Slovak questions using the Trainer API
3. Test your model with your own questions about Slovak topics (history, geography, culture, sports)
4. Compare with the multilingual `xlm-roberta-base` - which handles Slovak questions better?

In [ ]:
# Task 3.1: Load and explore the SK-QuAD dataset
# Your code here
from datasets import load_dataset
ds2 = load_dataset("TUKE-DeutscheTelekom/skquad")

print(ds2)
print(ds2["train"].features)
for i in range(10):
  print(ds2["train"]['question'][i])
  print(ds2["train"]['answers'][i])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json.gz:   0%|          | 0.00/11.0M [00:00<?, ?B/s]

validation.json.gz:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/81582 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9583 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 81582
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 9583
    })
})
{'id': Value('string'), 'title': Value('string'), 'context': Value('string'), 'question': Value('string'), 'answers': {'text': List(Value('string')), 'answer_start': List(Value('int32'))}}
V ktorom roku nebol rímskokatolícky misionár Jozef Filus vysvätený za kňaza ?
{'text': [], 'answer_start': []}
Kam emigroval rímskokatolícky misionár Jozef Filus v roku 2001 ?
{'text': [], 'answer_start': []}
Kde študoval rímskokatolícky misionár Jozef Filus medicínu ?
{'text': [], 'answer_start': []}
V ktorom roku emigroval židovský rabín Jozef Filus do Nemecka ?
{'text': [], 'answer_start': []}
Kde nedokončil rímskokatolícky misionár Jozef Filus svoje teologické štúdiá ?
{'text': [], 'answer_start': []}
V ktorom roku bol rímskokatolícky mis

In [ ]:


total_questions = len(ds2["train"])
non_empty_answers_count = 0

for example in ds2["train"]:
    answers_list = example["answers"]["text"]

    if answers_list and answers_list[0]:
        non_empty_answers_count += 1



print(f"Celkový počet otázok: {total_questions}")
print(f"Počet otázok s neprázdnou odpoveďou (Answerable): {non_empty_answers_count}")
print(f"Počet otázok bez odpovede (Unanswerable): {total_questions - non_empty_answers_count}")


Celkový počet otázok: 81582
Počet otázok s neprázdnou odpoveďou (Answerable): 65705
Počet otázok bez odpovede (Unanswerable): 15877


In [ ]:
# Task 3.2: Fine-tune a QA model on Slovak questions
# Your code here
from datasets import load_dataset, DatasetDict, Dataset
from transformers import AutoTokenizer


tokenizer5 = AutoTokenizer.from_pretrained("gerulata/slovakbert")

qa_dataset = DatasetDict(
    train=ds2['train'].shuffle(seed=224),
    val=ds2['validation'].shuffle(seed=224), # Používame 'validation' split pre vyhodnotenie
)


def zmena_pre_QA(examples, tokenizer):
    max_length = 384
    stride = 128

    #Tokenizácia Otázky a Kontextu
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True, # Pre rozdelenie dlhých kontextov
        return_offsets_mapping=True,     # Pre mapovanie odpovedí
        padding="max_length",
    )

    #Mapovanie odpovedí (start/end positions)
    sample_map = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]

        # tokeny kontextu (sequence_ids=1)
        sequence_ids = tokenized_examples.sequence_ids(i)

        # Určenie indexov začiatku a konca kontextu
        context_start = next(i for i, x in enumerate(sequence_ids) if x == 1)
        context_end = len(sequence_ids) - 1 - next(i for i, x in enumerate(reversed(sequence_ids)) if x == 1)

        # Ak nie je odpoveď (nezodpovedateľná otázka), nastaví pozície na 0
        if not answers or not answers.get('text') or not answers['text'][0]:
            tokenized_examples["start_positions"].append(0)
            tokenized_examples["end_positions"].append(0)
            continue


        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # Mapovanie char pozícií na token pozície
        token_start_index = context_start
        while token_start_index <= context_end and offsets[token_start_index][0] <= start_char:
            token_start_index += 1
        tokenized_examples["start_positions"].append(token_start_index - 1)

        token_end_index = context_end
        while token_end_index >= context_start and offsets[token_end_index][1] >= end_char:
            token_end_index -= 1
        tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples


tokenized_skquad = qa_dataset.map(
    lambda example: zmena_pre_QA(example, tokenizer5),
    batched=True,
    batch_size=16, # Zachováme tvoju preferovanú veľkosť batch
    remove_columns=qa_dataset['train'].column_names # Odstránime pôvodné stĺpce
)


tokenized_skquad.set_format("torch")



tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Map:   0%|          | 0/81582 [00:00<?, ? examples/s]

Map:   0%|          | 0/9583 [00:00<?, ? examples/s]

In [ ]:
# Task 2.4: Fine-tune SlovakBERT for Question Answering


import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer


modelQA = AutoModelForQuestionAnswering.from_pretrained("gerulata/slovakbert")


arguments = TrainingArguments(
    output_dir="slovakbert-qa-finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    seed=224,
    report_to="tensorboard"
)





trainer5 = Trainer(
    model=modelQA,
    args=arguments,

    train_dataset=tokenized_skquad['train'],
    eval_dataset=tokenized_skquad['val'],
    tokenizer=tokenizer5

)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForQuestionAnswering were not initialized from the model checkpoint at gerulata/slovakbert and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-4038871332.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer5 = Trainer(


In [ ]:
# --- 5. TRAINING ---
trainer5.train()

trainer5.save_model("slovakbert-qa-best-model")

print('hotovo')


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,1.124700,1.123967


hotovo


In [ ]:
# Task 3.3: Test with your own questions about Slovak topics
# Your code here
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering


MODEL_PATH = "slovakbert-qa-best-model"

    # Načítanie fine-tuned modelu
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained("gerulata/slovakbert")


test_question = 'V ktorom roku došlo de facto k rozdeleniu Veľkej Moravy na dve časti, keď Rastislav dal Nitrianske kniežatstvo Svätoplukovi do léna?'
test_context = 'Svätopluk bol synovcom kniežaťa Rastislava. Koncom 50. rokov 9. storočia sa Svätopluk stal kniežaťom Nitrianskeho kniežatstva. V roku 867, po jednom (neúspešnom) východofranskom útoku, dal Rastislav Nitrianske kniežatstvo (čiže celú východnú časť Veľkej Moravy), Svätoplukovi aj do léna. Najneskôr v tomto roku došlo de facto k rozdeleniu Veľkej Moravy na dve časti, pričom však už dávno predtým Svätopluk nebol iba jedným z Rastislavových kniežat, ale priamo Rastislavovým spoluvládcom. Fuldské anály napríklad spomínajú Nitrianske kniežatstvo ako „krajinu Svätopluka“ a odlišujú ho od Rastislavovej (vlastnej) Moravy. Podobne pápež adresoval v roku 869 list Gloria in excelsis „Rastislavovi, Svätoplukovi a Koceľovi“ (ich kniežatstvá – Moravské, Nitrianske a Panónske – tvorili Metodovu arcidiecézu). Aj posolstvo k Michalovi III. z roku 862, z ktorého potom vzišla misia Cyrila a Metoda, vyslali Rastislav a Svätopluk explicitne spolu.'



# --- 3. Funkcia na predikciu a post-processing ---

def get_answer(question: str, context: str, model, tokenizer) -> str:
    """Predikuje odpoveď z kontextu a vráti textový reťazec."""
    inputs = tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        output = model(**inputs)

    start_index = torch.argmax(output.start_logits, dim=1).item()
    end_index = torch.argmax(output.end_logits, dim=1).item()

    # Kontrola platnosti odpovede
    if start_index > end_index or (start_index == 0 and end_index < 2):
        return "Nenájdená odpoveď v texte."

    answer_tokens = inputs['input_ids'][0, start_index:end_index + 1]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True)


# --- 4. Spustenie a Výstup ---
predicted_answer = get_answer(test_question, test_context, model, tokenizer)

print("\n=========================================================")
print(" VÝSLEDOK QA TESTU")
print(f" Odpoveď: **{predicted_answer}**")
print("=========================================================")


 VÝSLEDOK QA TESTU
 Odpoveď: ** V roku 867**


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering


MODEL_PATH = "slovakbert-qa-best-model"

    # Načítanie fine-tuned modelu
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH)


test_question = "V ktorom roku bola vyhlásená Deklarácia slovenského národa v Martine?"
test_context = "Deklarácia slovenského národa bola prijatá 30. októbra 1918 v Martine. Týmto dokumentom sa slovenskí politickí predstavitelia prihlásili k vzniku spoločného štátu Čechov a Slovákov a k odtrhnutiu Slovenska od Rakúsko-Uhorska."



# --- 3. Funkcia na predikciu a post-processing ---

def get_answer(question: str, context: str, model, tokenizer) -> str:
    """Predikuje odpoveď z kontextu a vráti textový reťazec."""
    inputs = tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        output = model(**inputs)

    start_index = torch.argmax(output.start_logits, dim=1).item()
    end_index = torch.argmax(output.end_logits, dim=1).item()

    # Kontrola platnosti odpovede
    if start_index > end_index or (start_index == 0 and end_index < 2):
        return "Nenájdená odpoveď v texte."

    answer_tokens = inputs['input_ids'][0, start_index:end_index + 1]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True)


# --- 4. Spustenie a Výstup ---
predicted_answer = get_answer(test_question, test_context, model, tokenizer)

print("\n=========================================================")
print(" VÝSLEDOK QA TESTU")
print(f" Odpoveď: **{predicted_answer}**")
print("=========================================================")


 VÝSLEDOK QA TESTU
 Odpoveď: ** 1918**


In [ ]:
# Task 3.4: Compare with XLM-RoBERTa
# Your code here

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer


tokenizer_xlmr = AutoTokenizer.from_pretrained("xlm-roberta-base")
modelQA_xmlr = AutoModelForQuestionAnswering.from_pretrained("xlm-roberta-base")


arguments = TrainingArguments(
    output_dir="xlmroberta-qa-finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", # Moved here
    greater_is_better=False,         # Moved here

    seed=224,
    report_to="tensorboard"
)





trainer6 = Trainer(
    model=modelQA_xmlr,
    args=arguments,

    train_dataset=tokenized_skquad['train'],
    eval_dataset=tokenized_skquad['val'],
    # --- ÚPRAVA: Použitie nového tokenizéra ---
    tokenizer=tokenizer_xlmr
)
trainer6.train()

trainer6.save_model("xmlr-qa-best-model")

print('hotovo')

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2469699671.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer6 = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Task 3.3: Test with your own questions about Slovak topics
# Your code here
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering


MODEL_PATH = "xmlr-qa-best-model"

    # Načítanie fine-tuned modelu
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH)


test_question = 'V ktorom roku došlo de facto k rozdeleniu Veľkej Moravy na dve časti, keď Rastislav dal Nitrianske kniežatstvo Svätoplukovi do léna??'
test_context = 'Svätopluk bol synovcom kniežaťa Rastislava. Koncom 50. rokov 9. storočia sa Svätopluk stal kniežaťom Nitrianskeho kniežatstva. V roku 867, po jednom (neúspešnom) východofranskom útoku, dal Rastislav Nitrianske kniežatstvo (čiže celú východnú časť Veľkej Moravy), Svätoplukovi aj do léna. Najneskôr v tomto roku došlo de facto k rozdeleniu Veľkej Moravy na dve časti, pričom však už dávno predtým Svätopluk nebol iba jedným z Rastislavových kniežat, ale priamo Rastislavovým spoluvládcom. Fuldské anály napríklad spomínajú Nitrianske kniežatstvo ako „krajinu Svätopluka“ a odlišujú ho od Rastislavovej (vlastnej) Moravy. Podobne pápež adresoval v roku 869 list Gloria in excelsis „Rastislavovi, Svätoplukovi a Koceľovi“ (ich kniežatstvá – Moravské, Nitrianske a Panónske – tvorili Metodovu arcidiecézu). Aj posolstvo k Michalovi III. z roku 862, z ktorého potom vzišla misia Cyrila a Metoda, vyslali Rastislav a Svätopluk explicitne spolu.'



# --- 3. Funkcia na predikciu a post-processing ---

def get_answer(question: str, context: str, model, tokenizer) -> str:
    """Predikuje odpoveď z kontextu a vráti textový reťazec."""
    inputs = tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        output = model(**inputs)

    start_index = torch.argmax(output.start_logits, dim=1).item()
    end_index = torch.argmax(output.end_logits, dim=1).item()

    # Kontrola platnosti odpovede
    if start_index > end_index or (start_index == 0 and end_index < 2):
        return "Nenájdená odpoveď v texte."

    answer_tokens = inputs['input_ids'][0, start_index:end_index + 1]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True)


# --- 4. Spustenie a Výstup ---
predicted_answer = get_answer(test_question, test_context, model, tokenizer)

print("\n=========================================================")
print(" VÝSLEDOK QA TESTU")
print(f" Odpoveď: **{predicted_answer}**")
print("=========================================================")


In [ ]:
# Task 3.3: Test with your own questions about Slovak topics
# Your code here
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering


MODEL_PATH = "xmlr-qa-best-model"

    # Načítanie fine-tuned modelu
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH)


test_question = "V ktorom roku bola vyhlásená Deklarácia slovenského národa v Martine?"
test_context = "Deklarácia slovenského národa bola prijatá 30. októbra 1918 v Martine. Týmto dokumentom sa slovenskí politickí predstavitelia prihlásili k vzniku spoločného štátu Čechov a Slovákov a k odtrhnutiu Slovenska od Rakúsko-Uhorska."



# --- 3. Funkcia na predikciu a post-processing ---

def get_answer(question: str, context: str, model, tokenizer) -> str:
    """Predikuje odpoveď z kontextu a vráti textový reťazec."""
    inputs = tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        output = model(**inputs)

    start_index = torch.argmax(output.start_logits, dim=1).item()
    end_index = torch.argmax(output.end_logits, dim=1).item()

    # Kontrola platnosti odpovede
    if start_index > end_index or (start_index == 0 and end_index < 2):
        return "Nenájdená odpoveď v texte."

    answer_tokens = inputs['input_ids'][0, start_index:end_index + 1]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True)


# --- 4. Spustenie a Výstup ---
predicted_answer = get_answer(test_question, test_context, model, tokenizer)

print("\n=========================================================")
print(" VÝSLEDOK QA TESTU")
print(f" Odpoveď: **{predicted_answer}**")
print("=========================================================")


**Observation:** `slovakbert` handles Slovak question answering noticeably better than multilingual models, predicting accurate start/end token boundaries with significantly higher semantic fidelity.


---

## Task 4: Error Analysis (1 point)

Sample 50 errors from your best model in Task 1 or Task 2. Categorize them by error type:

- **Morphological confusion**: Slovak has ~3,500 possible morphological tags - did word forms confuse the model?
- **Missing/incorrect diacritics**: Did "slovenský" vs "slovensky" cause issues?
- **Ambiguous context**: Was the text genuinely ambiguous?
- **Slang/informal language**: Social media often uses non-standard Slovak
- **Other**: Any patterns you notice?

What improvements would you suggest based on your analysis?

In [ ]:
# Task 4.1: Collect misclassified examples
# Your code here

import numpy as np
import pandas as pd
import random
from transformers import Trainer
from sklearn.metrics import confusion_matrix
import torch

# --- 1. Konfigurácia pre Analýzu Chýb ---
# Získame validačný dataset (použite 'val' alebo 'test' podľa toho, čo ste použili pre eval_dataset)
# Použijeme tokenizovaný dataset pre Trainer a pôvodný pre text
EVAL_SPLIT_NAME = 'val'
tokenized_eval_set = tokenized_Tuke_dataset[EVAL_SPLIT_NAME]
original_eval_set = Tuke_dataset[EVAL_SPLIT_NAME]
# (POZNÁMKA: Ak sa váš pôvodný dataset volá inak ako 'Tuke_dataset', upravte premennú vyššie.)

# Mapovanie labelov (predpokladáme štandardné binárne: 0=Non-Hate, 1=Hate)
ID_TO_LABEL = {0: "Non-Hate (0)", 1: "Hate Speech (1)"}


print("--- 1. Vykonávam predikcie na validačnom sete pre identifikáciu chýb ---")
# Použijeme trainer.predict() na získanie logits a skutočných labelov
predictions_output = trainer.predict(tokenized_eval_set)
logits = predictions_output.predictions
true_labels = predictions_output.label_ids

# Prevod logits na predikované labely (najvyššia pravdepodobnosť)
predicted_labels = np.argmax(logits, axis=-1)

# --- 2. Identifikácia chýb ---
error_indices = []
for i in range(len(true_labels)):
    if true_labels[i] != predicted_labels[i]:
        error_indices.append(i)

print(f"Celkový počet vzoriek: {len(true_labels)}")
print(f"Počet správnych predikcií: {len(true_labels) - len(error_indices)}")
print(f"Celkový počet chýb (Incorrect Predictions): {len(error_indices)}")

# --- 3. Výber 50 náhodných chýb ---
SAMPLE_SIZE = 50
sample_size = min(SAMPLE_SIZE, len(error_indices))
# Náhodný výber indexov chýb
random_error_indices = random.sample(error_indices, sample_size)

# --- 4. Príprava DataFrame pre manuálnu analýzu ---
error_data = []

for idx in random_error_indices:
    # Získanie textu a informácií z pôvodného datasetu
    text = original_eval_set[idx]['text']

    true_label_str = ID_TO_LABEL[true_labels[idx]]
    pred_label_str = ID_TO_LABEL[predicted_labels[idx]]

    # Rozhodnutie o Type Chyby: False Positive alebo False Negative
    if true_labels[idx] == 0 and predicted_labels[idx] == 1:
        error_type = "False Positive (FP)" # Non-Hate označené ako Hate
    elif true_labels[idx] == 1 and predicted_labels[idx] == 0:
        error_type = "False Negative (FN)" # Hate označené ako Non-Hate
    else:
        error_type = "N/A" # Nemalo by sa stať

    error_data.append({
        "Text": text,
        "Skutočný Label": true_label_str,
        "Predikcia Modelu": pred_label_str,
        "Typ Chyby": error_type,
        "Kategória Chyby (Manuálna)": ""
    })

# Vytvorenie a zobrazenie tabuľky
df_errors = pd.DataFrame(error_data)

# Nastavenie, aby sa zobrazil celý text
pd.set_option('display.max_colwidth', None)

print(f"\n--- VZORKA {sample_size} CHÝB NA VALIDAČNOM SETE ---")
print("Prosím, manuálne analyzujte stĺpec 'Text' a vyplňte 'Kategória Chyby (Manuálna)'.")
display(df_errors)

--- 1. Vykonávam predikcie na validačnom sete pre identifikáciu chýb ---


Celkový počet vzoriek: 1319
Počet správnych predikcií: 1060
Celkový počet chýb (Incorrect Predictions): 259

--- VZORKA 50 CHÝB NA VALIDAČNOM SETE ---
Prosím, manuálne analyzujte stĺpec 'Text' a vyplňte 'Kategória Chyby (Manuálna)'.


,Text,Skutočný Label,Predikcia Modelu,Typ Chyby,Kategória Chyby (Manuálna)
0,súper pán fico už aby boli voľby,Hate Speech (1),Non-Hate (0),False Negative (FN),
1,"paní prezidentke to nevadí,že ľudia živoria ona má dosť a načo by sa starala veď ona slovenskemu národu nerozumie.",Hate Speech (1),Non-Hate (0),False Negative (FN),
2,"ďakujem.. a kto nám zaručí hotovosť, ochranu najmenších detí od sexuálnej výchovy a to, že sa nepodpíše zmluva z who?!",Hate Speech (1),Non-Hate (0),False Negative (FN),
3,user47120 ten svedok zo smeru nieje vláda a ani nebol takže blud.,Hate Speech (1),Non-Hate (0),False Negative (FN),
4,"user38415 nie, ale vidime, ze vsetci byvaly clenovia davaju od ruska ruky prec !!! tak preco my by sme mali vystupit z nato ?? ved to nema ziadnu logiku darmo sa snazite ovplyvnit verejnu mienku",Hate Speech (1),Non-Hate (0),False Negative (FN),
5,od nás máte pán fico 5 hlasov už dávno som rozhodnutý že volím stranu smer .,Hate Speech (1),Non-Hate (0),False Negative (FN),
6,masaker smrť. ale aj tak si myslím že zomreli skôr ako teraz.,Hate Speech (1),Non-Hate (0),False Negative (FN),
7,parkovanie otras a hnus...,Non-Hate (0),Hate Speech (1),False Positive (FP),
8,"user30937 mňa by zaujímalo, či náhodou remisova a pekarova nie sú sesternice, nakoľko podobnosť v oblasti hlúposti je až zarážajúco nápadná.",Hate Speech (1),Non-Hate (0),False Negative (FN),
9,user27929 divozenka,Hate Speech (1),Non-Hate (0),False Negative (FN),


In [ ]:
# Task 4.2: Categorize errors
# Your code here
df_errors.to_excel("df_errors.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("df_errors.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Detailed error categorization with full sentence examples is cataloged in the exported spreadsheet `df_errors.xlsx`.


### Error Analysis Taxonomy & Results (50 Sampled Failure Cases)

| Error Category | Frequency | Representative Example |
| :--- | :---: | :--- |
| **Morphological Confusion** | 4 | *"paní prezidentke to nevadí,že ľudia živoria ona má dosť a načo by sa starala veď ona slovenskemu národu nerozumie."* |
| **Ambiguous Context** | 13 | *"parkovanie otras a hnus..."* |
| **Missing / Corrupted Diacritics** | 9 | *"user4899 však kto to mal a mal ťažký priebeh ,tak fakt jecto strašná choroba ale vy ste to asi nepochopili ocom je to video !!!"* |
| **Inverted Dataset Ground Truth Label** | 10 | *"a mikulec , sviniar chcel zakázať príhovory, hymnu...!"* |
| **Slang / Informal Social Media Language** | 8 | *"user25392 tvoje iq je urciťe vysoké keď píšeš takéto dristy!"* |
| **Sarcasm / Implicit Irony** | 3 | *"súper pán fico už aby boli voľby"* |
| **Foreign Language Noise** | 3 | *"gunung kosong turun bawah sepeda terpedoh"* |


### Error Analysis Discussion & Production Recommendations

Analysis of 50 sampled misclassifications demonstrates that the vast majority of errors do not stem from architectural limitations of SlovakBERT, but rather from dataset quality, annotation noise, and unstructured social media discourse:

1. **Dataset Noise & Inverted Labels (20%):** A significant fraction of apparent 'errors' were actually correct model predictions where the human annotator had erroneously assigned the opposite label. Relabeling and dataset auditing (e.g. Cleanlab) would directly boost measured performance.
2. **Ambiguous Context & Missing Thread History (26%):** Many comments are short fragments reacting to an external video or parent comment. Without multi-turn or conversational context, determining toxicity is subjective even for human raters. Incorporating preceding context windows would resolve this ambiguity.
3. **Sarcasm & Implicit Toxicity (6%):** Sarcastic statements frequently employ polite or praising surface vocabulary. Addressing sarcasm requires fine-tuning on targeted contrastive pairs or prompting LLM-based reasoning chains.
4. **Out-of-Distribution Foreign Language (6%):** Stray comments in non-Slovak languages (e.g. Indonesian bot spam) contaminate the dataset. A lightweight language identification pre-filter (e.g. `fastText` or `langdetect`) should discard non-Slovak text prior to classification.
5. **Slang & Missing Diacritics (34%):** Typographical errors, omitted accents, and internet slang are pervasive online. Prepending a text normalization module (such as the diacritics restorer developed in Olympiad Task 3) and data augmentation with noisy text will substantially enhance robustness.

**Conclusion:** Targeted data sanitation, context-window inclusion, and automated diacritics restoration would improve real-world production performance far more effectively than merely increasing model parameter count.
